In [6]:
# --- SETUP: paths + robust pairing + I/O helpers (run once) ---
from pathlib import Path
import re, os, json, math
import numpy as np
import nibabel as nib
from scipy.ndimage import gaussian_filter, zoom, rotate, shift, binary_closing, generate_binary_structure, label

# SOURCE hires test set (single-folder mode with images + masks)
SRC_DIR   = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires")
# Where to place all degraded datasets
OUT_ROOT  = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# --------- pairing (mirrors your training loader’s spirit) ----------
def is_mask_name(name: str) -> bool:
    n = name.lower()
    return ("mask" in n) or ("lesion" in n)

def strip_ext(name: str) -> str:
    return name[:-7] if name.endswith(".nii.gz") else os.path.splitext(name)[0]

def norm_key(name: str) -> str:
    # remove known suffixes to match img<->mask
    stem = strip_ext(name)
    # drop common endings
    for sfx in ["_T1w_MNI_norm","_T1w_MNI","_T1w_brain","_T1w","_T1","_image","_img","_img_prepped"]:
        if stem.endswith(sfx): stem = stem[: -len(sfx)]
    for sfx in ["_lesion_mask_MNI_clean","_lesion_mask_MNI","_lesion_mask","_desc-lesion_mask","_mask","_mask_prepped"]:
        if stem.endswith(sfx): stem = stem[: -len(sfx)]
    return stem.rstrip("_")

# build maps
imgs, msks = {}, {}
for p in sorted(SRC_DIR.glob("*.nii.gz")):
    (msks if is_mask_name(p.name) else imgs)[norm_key(p.name)] = p

keys = sorted(set(imgs) & set(msks))
pairs = [(imgs[k], msks[k]) for k in keys]
print(f"Found pairs: {len(pairs)}  (imgs={len(imgs)}, msks={len(msks)})")

# --------- helpers ----------
def load_nii(p: Path) -> nib.Nifti1Image:
    return nib.load(str(p))

def data_f32(img: nib.Nifti1Image) -> np.ndarray:
    return np.asarray(img.get_fdata(dtype=np.float32), dtype=np.float32)

def save_like(ref_img: nib.Nifti1Image, array: np.ndarray, out_path: Path, dtype=None):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    arr = (array.astype(dtype) if dtype is not None else array.astype(np.float32))
    nii = nib.Nifti1Image(arr, ref_img.affine, ref_img.header.copy())
    nib.save(nii, str(out_path))

def save_and_update_spacing(ref_img: nib.Nifti1Image, array: np.ndarray, out_path: Path, new_spacing_xyz):
    """Use ref affine but update pixdim to reflect a new voxel size (mm)."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    hdr = ref_img.header.copy()
    hdr["pixdim"][1:4] = np.array(new_spacing_xyz, dtype=np.float32)
    nii = nib.Nifti1Image(array.astype(np.float32), ref_img.affine, hdr)
    nib.save(nii, str(out_path))

def resample_factor(arr, factors, order):
    """Resample by 1/factors using scipy.zoom (anti-alias BEFORE calling)."""
    # zoom expects output/input; if factor=2 (coarsen), zoom=1/2
    zf = tuple(1.0/f for f in factors)
    return zoom(arr, zf, order=order, prefilter=True)

def ensure_uint8_mask(b):
    return (b > 0).astype(np.uint8)

def percentile_scale(vol, pmin=1, pmax=99):
    lo, hi = np.percentile(vol, [pmin, pmax])
    hi = max(hi, lo + 1e-6)
    x = np.clip((vol - lo) / (hi - lo), 0, 1)
    return x

def _is_top_level(p: Path) -> bool:
    # treat files directly inside SRC_DIR as "top-level", not in subfolders like t1/, masks/
    try:
        return p.parent.resolve() == SRC_DIR.resolve()
    except Exception:
        return False

def discover_pairs(src_dir: Path):
    """
    Recursively scan SRC_DIR for images/masks (flat + subfolders),
    but keep only one image and one mask per norm_key.
    Preference: choose top-level files over nested ones if duplicates exist.
    """
    # Collect candidates
    img_cands, msk_cands = {}, {}
    for p in src_dir.rglob("*.nii.gz"):
        key = norm_key(p.name)
        if not key:
            continue
        if is_mask_name(p.name):
            bucket = msk_cands
        else:
            bucket = img_cands

        # If key not seen yet, take this file.
        # If key seen, prefer top-level; if both top-level or both nested, keep the first seen.
        if key not in bucket:
            bucket[key] = p
        else:
            keep = bucket[key]
            if _is_top_level(p) and not _is_top_level(keep):
                bucket[key] = p  # prefer top-level
            # else keep existing

    keys = sorted(set(img_cands) & set(msk_cands))
    pairs = [(img_cands[k], msk_cands[k]) for k in keys]
    return pairs


Found pairs: 138  (imgs=138, msks=138)


# 1) Crude downsampling (no anti-alias) — “worst-case decimation”

Why / real-world: Stress-tests robustness to botched resampling pipelines that skip anti-aliasing, producing jaggies and aliasing. Rare, but if your preproc ever fails, this is what it looks like.

## Crude downsampling → crude upsampling (2×2×5)

**What this does**
- **Decimate** the volume by fixed factors (e.g., 2× in X/Y and 5× in Z) via simple voxel picking (no anti-aliasing).
- **Nearest-repeat upsample** back to the original matrix size (repeat along each axis), then **center pad/crop** to match the exact shape.
- **Save on the original 1 mm grid** (affine/header unchanged), so spacing/pixdim remain the same while *effective* resolution is degraded.

**Why do it this way?**
- Keeps geometry identical to your evaluation set (no resampling surprises downstream).
- Emulates severe partial-volume and aliasing artifacts that arise from coarse acquisitions without regridding the header.

**What it mimics**
- Low through-plane resolution and thick slices (factor 5 in Z).
- In-plane coarsening (factor 2 in X/Y) without any anti-alias prefilter—i.e., a “worst-case” crude reconstruction.

**Caveats**
- No anti-aliasing before decimation → intentionally introduces aliasing/ringing.
- Nearest repetition back to full size creates blocky edges; not a physically accurate recon, just a stress test.
- Because header spacing is unchanged, the images *look* like 1 mm voxels but have degraded information content.

**Useful knobs**
- `factors = (fx, fy, fz)`: coarsening per axis (e.g., `(2,2,5)`).
- Swap in anti-aliasing (e.g., `gaussian_filter`) before decimation if you want a “less harsh” variant.
- Replace nearest-repeat with linear/b-spline upsampling for different artifact profiles (still keep final matrix the same).


In [7]:
# === Crude low-res simulation (dedup-aware): decimate -> nearest upsample back to original shape ===
from pathlib import Path
import numpy as np
import nibabel as nib

# ------- config -------
FACTORS   = (2, 2, 5)  # (x,y,z) crude coarsening
OVERWRITE = True      # set True to re-generate outputs even if they exist
# Use your existing SRC_DIR and OUT_ROOT from the setup cell
OUT_DIR   = OUT_ROOT / "test_hires_crude_2x2x5x_backTo1mm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _is_top_level(p: Path) -> bool:
    # treat files directly inside SRC_DIR as "top-level", not in subfolders like t1/, masks/
    try:
        return p.parent.resolve() == SRC_DIR.resolve()
    except Exception:
        return False

def discover_pairs(src_dir: Path):
    """
    Recursively scan SRC_DIR for images/masks (flat + subfolders),
    but keep only one image and one mask per norm_key.
    Preference: choose top-level files over nested ones if duplicates exist.
    """
    # Collect candidates
    img_cands, msk_cands = {}, {}
    for p in src_dir.rglob("*.nii.gz"):
        key = norm_key(p.name)
        if not key:
            continue
        if is_mask_name(p.name):
            bucket = msk_cands
        else:
            bucket = img_cands

        # If key not seen yet, take this file.
        # If key seen, prefer top-level; if both top-level or both nested, keep the first seen.
        if key not in bucket:
            bucket[key] = p
        else:
            keep = bucket[key]
            if _is_top_level(p) and not _is_top_level(keep):
                bucket[key] = p  # prefer top-level
            # else keep existing

    keys = sorted(set(img_cands) & set(msk_cands))
    pairs = [(img_cands[k], msk_cands[k]) for k in keys]
    return pairs

def decimate_crude(arr: np.ndarray, fx, fy, fz) -> np.ndarray:
    # integer sub-sampling (aliasing intentionally preserved)
    return arr[::fx, ::fy, ::fz]

def nearest_repeat(arr: np.ndarray, fx, fy, fz) -> np.ndarray:
    # zero-order hold upsampling back to large matrix (blocky)
    out = np.repeat(arr, fx, axis=0)
    out = np.repeat(out, fy, axis=1)
    out = np.repeat(out, fz, axis=2)
    return out

def pad_or_crop_to(arr: np.ndarray, target_shape):
    """Pad with edge values or crop centrally to hit exact target shape."""
    out = arr
    for ax in range(3):
        cur = out.shape[ax]
        tgt = target_shape[ax]
        if cur == tgt:
            continue
        if cur > tgt:
            # centered crop
            start = (cur - tgt)//2
            sl = [slice(None), slice(None), slice(None)]
            sl[ax] = slice(start, start+tgt)
            out = out[tuple(sl)]
        else:
            # pad by repeating edge values to maintain blockiness
            pad_before = (tgt - cur)//2
            pad_after  = tgt - cur - pad_before
            pads = [(0,0)]*out.ndim
            pads[ax] = (pad_before, pad_after)
            out = np.pad(out, pads, mode="edge")
    return out

# ---- run ----
pairs = discover_pairs(SRC_DIR)
print(f"Discovered unique pairs: {len(pairs)} (deduped across flat + subfolders)")

fx, fy, fz = FACTORS
wrote = 0
for i, (img_p, msk_p) in enumerate(pairs, 1):
    img_ref = load_nii(img_p)       # original 1mm image (for shape/affine/header)
    msk_ref = load_nii(msk_p)
    x = data_f32(img_ref)
    y = data_f32(msk_ref)

    # crude downsample
    xd = decimate_crude(x, fx, fy, fz)
    yd = decimate_crude(y, fx, fy, fz)

    # crude upsample (nearest repeat) back to original matrix size
    xu = nearest_repeat(xd, fx, fy, fz)
    yu = nearest_repeat(yd, fx, fy, fz)

    # pad/crop to match exactly
    target_shape = x.shape
    xu = pad_or_crop_to(xu, target_shape)
    yu = pad_or_crop_to(yu, target_shape)

    # output names (match your naming scheme)
    base = strip_ext(img_p.name).replace("_T1w_MNI_norm","").replace("_T1w","")
    out_img = OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz"
    out_msk = OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz"

    if not OVERWRITE and out_img.exists() and out_msk.exists():
        if i % 25 == 0 or i == len(pairs):
            print(f"[{i}/{len(pairs)}] (skip, exists) {out_img.name}")
        continue

    # save using the original affine/header so volumes are back on 1mm grid
    save_like(img_ref, xu.astype(np.float32), out_img, dtype=np.float32)
    save_like(img_ref, ensure_uint8_mask(yu), out_msk, dtype=np.uint8)
    wrote += 1

    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] wrote {out_img.name} & {out_msk.name}")

print(f"Done → {OUT_DIR} | wrote {wrote} case(s)")


Discovered unique pairs: 138 (deduped across flat + subfolders)
[10/138] wrote sub-M2086_ses-3021_T1w_MNI_norm.nii.gz & sub-M2086_ses-3021_lesion_mask_MNI_clean.nii.gz
[10/138] wrote sub-M2086_ses-3021_T1w_MNI_norm.nii.gz & sub-M2086_ses-3021_lesion_mask_MNI_clean.nii.gz
[20/138] wrote sub-M2131_ses-472_T1w_MNI_norm.nii.gz & sub-M2131_ses-472_lesion_mask_MNI_clean.nii.gz
[20/138] wrote sub-M2131_ses-472_T1w_MNI_norm.nii.gz & sub-M2131_ses-472_lesion_mask_MNI_clean.nii.gz
[30/138] wrote sub-M2198_ses-1073_T1w_MNI_norm.nii.gz & sub-M2198_ses-1073_lesion_mask_MNI_clean.nii.gz
[30/138] wrote sub-M2198_ses-1073_T1w_MNI_norm.nii.gz & sub-M2198_ses-1073_lesion_mask_MNI_clean.nii.gz
[40/138] wrote sub-M2238_ses-1374_T1w_MNI_norm.nii.gz & sub-M2238_ses-1374_lesion_mask_MNI_clean.nii.gz
[40/138] wrote sub-M2238_ses-1374_T1w_MNI_norm.nii.gz & sub-M2238_ses-1374_lesion_mask_MNI_clean.nii.gz
[50/138] wrote sub-M2306_ses-707_T1w_MNI_norm.nii.gz & sub-M2306_ses-707_lesion_mask_MNI_clean.nii.gz
[50/13

# 2) Thick slices (Z-only downsample with anti-alias)

Why / real-world: Very common: 1×1×5 mm or similar to shorten scans. Causes strong partial-volume in Z.

What the code does

Applies a Gaussian blur along Z only: gaussian_filter(x, sigma=(0,0,sigma_z)).

Then decimates in Z by an integer factor (e.g., 5): xr = xb[:, :, ::5].

Updates only the Z voxel spacing: new_dz = old_dz * 5.

Mask is decimated slice-wise using nearest (simple stride). No smoothing is applied to masks to keep labels crisp.

What this mimics

Common protocol trade-off: keep in-plane high (e.g., ~1×1 mm) but use thick slices (e.g., 5 mm) to shorten scan time.

Real images have through-plane blur and partial volume: structures smaller than the slice thickness smear into neighbors.

Why it’s useful

Many routine T1/T2/FLAIR stacks are anisotropic. Models trained on 1 mm isotropic can struggle here; this tests that gap.

Caveats

True slice thickness combines slice profile + gaps; here we simulate the net blur/gross thickness (a good approximation).

In [8]:
# === Thick-slice simulation: blur Z → decimate Z → nearest repeat back to original depth ===
from pathlib import Path
import numpy as np
import nibabel as nib
from scipy.ndimage import gaussian_filter

# ------- config -------
OUT_DIR    = OUT_ROOT / "test_hires_thickslice_1x1x5mm_backTo1mm"
factor_z   = 5          # coarsen only through-plane
sigma_z_vox = 2.0       # Gaussian blur along Z (in voxels) before decimation
OVERWRITE  = False

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Reuse: discover_pairs(SRC_DIR), load_nii, data_f32, ensure_uint8_mask, save_like, strip_ext, pad_or_crop_to

pairs = discover_pairs(SRC_DIR)
print(f"Discovered unique pairs: {len(pairs)} (deduped across flat + subfolders)")

wrote = 0
for i, (img_p, msk_p) in enumerate(pairs, 1):
    img_ref = load_nii(img_p)
    msk_ref = load_nii(msk_p)
    x = data_f32(img_ref)
    y = data_f32(msk_ref)

    # 1) blur only along Z to mimic slice thickness PSF
    xb = gaussian_filter(x, sigma=(0.0, 0.0, float(sigma_z_vox)), mode="nearest")

    # 2) decimate Z (keep XY)
    xd = xb[:, :, ::factor_z]
    yd = y[:,  :, ::factor_z]           # masks: no blur, just decimate

    # 3) crude upsample along Z back to original depth (nearest repeat)
    xu = np.repeat(xd, factor_z, axis=2)
    yu = np.repeat(yd, factor_z, axis=2)

    # 4) pad/crop in Z (and safeguard XY) to exact original shape
    target_shape = x.shape
    xu = pad_or_crop_to(xu, target_shape)
    yu = pad_or_crop_to(yu, target_shape)

    # 5) write with original affine/header so we’re back on 1mm grid
    base = strip_ext(img_p.name).replace("_T1w_MNI_norm","").replace("_T1w","")
    out_img = OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz"
    out_msk = OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz"

    if not OVERWRITE and out_img.exists() and out_msk.exists():
        if i % 25 == 0 or i == len(pairs):
            print(f"[{i}/{len(pairs)}] (skip, exists) {out_img.name}")
        continue

    save_like(img_ref, xu.astype(np.float32), out_img, dtype=np.float32)
    save_like(img_ref, ensure_uint8_mask(yu), out_msk, dtype=np.uint8)
    wrote += 1

    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] wrote {out_img.name} & {out_msk.name}")

print(f"Done → {OUT_DIR} | wrote {wrote} case(s)")


Discovered unique pairs: 138 (deduped across flat + subfolders)
[10/138] wrote sub-M2086_ses-3021_T1w_MNI_norm.nii.gz & sub-M2086_ses-3021_lesion_mask_MNI_clean.nii.gz
[10/138] wrote sub-M2086_ses-3021_T1w_MNI_norm.nii.gz & sub-M2086_ses-3021_lesion_mask_MNI_clean.nii.gz
[20/138] wrote sub-M2131_ses-472_T1w_MNI_norm.nii.gz & sub-M2131_ses-472_lesion_mask_MNI_clean.nii.gz
[20/138] wrote sub-M2131_ses-472_T1w_MNI_norm.nii.gz & sub-M2131_ses-472_lesion_mask_MNI_clean.nii.gz
[30/138] wrote sub-M2198_ses-1073_T1w_MNI_norm.nii.gz & sub-M2198_ses-1073_lesion_mask_MNI_clean.nii.gz
[30/138] wrote sub-M2198_ses-1073_T1w_MNI_norm.nii.gz & sub-M2198_ses-1073_lesion_mask_MNI_clean.nii.gz
[40/138] wrote sub-M2238_ses-1374_T1w_MNI_norm.nii.gz & sub-M2238_ses-1374_lesion_mask_MNI_clean.nii.gz
[40/138] wrote sub-M2238_ses-1374_T1w_MNI_norm.nii.gz & sub-M2238_ses-1374_lesion_mask_MNI_clean.nii.gz
[50/138] wrote sub-M2306_ses-707_T1w_MNI_norm.nii.gz & sub-M2306_ses-707_lesion_mask_MNI_clean.nii.gz
[50/13

# 3) In-plane coarsening (XY downsample; Z intact)

Why / real-world: Protocols that keep thin slices but coarsen in-plane to reduce time. Tests sensitivity to small in-plane structures.

What the code does

Applies a Gaussian blur in X/Y: gaussian_filter(x, sigma=(sigma_xy, sigma_xy, 0)).

Decimates in X/Y by an integer factor (e.g., 2): xr = xb[::2, ::2, :].

Updates only X/Y spacing; Z spacing unchanged.

Mask is downsampled with nearest in X/Y by the same stride.

What this mimics

Protocols that keep thin slices but reduce the in-plane matrix (e.g., 256→128) to cut scan time or extend coverage.

Small cortical or juxtacortical lesions get “blockier” and less distinct in-plane.

Why it’s useful

Tests sensitivity to in-plane resolution loss separately from slice thickness effects.

Caveats

Real recon often uses vendor-specific filters; our Gaussian + decimate is a principled, reproducible stand-in.

In [9]:
# === In-plane coarsening: blur XY → decimate XY → nearest-repeat back to original matrix ===
from pathlib import Path
import numpy as np
import nibabel as nib
from scipy.ndimage import gaussian_filter

# ------- config -------
OUT_DIR    = OUT_ROOT / "test_hires_inplane_2x2x1mm_backTo1mm"
factor_xy  = 2           # coarsen only in-plane
sigma_xy   = 1.0         # pre-blur in X/Y (voxels) before decimation
OVERWRITE  = False

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Reuse: discover_pairs(SRC_DIR), load_nii, data_f32, ensure_uint8_mask, save_like, strip_ext, pad_or_crop_to

pairs = discover_pairs(SRC_DIR)
print(f"Discovered unique pairs: {len(pairs)} (deduped across flat + subfolders)")

wrote = 0
for i, (img_p, msk_p) in enumerate(pairs, 1):
    img_ref = load_nii(img_p)
    msk_ref = load_nii(msk_p)
    x = data_f32(img_ref)
    y = data_f32(msk_ref)

    # 1) Anti-alias in-plane only
    xb = gaussian_filter(x, sigma=(float(sigma_xy), float(sigma_xy), 0.0), mode="nearest")

    # 2) Decimate X/Y
    xd = xb[::factor_xy, ::factor_xy, :]
    yd = y [::factor_xy, ::factor_xy, :]   # masks: nearest in XY (no blur)

    # 3) Crude upsample back to original matrix with nearest (repeat) in X/Y
    xu = np.repeat(np.repeat(xd, factor_xy, axis=0), factor_xy, axis=1)
    yu = np.repeat(np.repeat(yd, factor_xy, axis=0), factor_xy, axis=1)

    # 4) Pad/crop to match exactly (handles odd sizes / non-divisible dims)
    target_shape = x.shape
    xu = pad_or_crop_to(xu, target_shape)
    yu = pad_or_crop_to(yu, target_shape)

    # 5) Save on original 1mm grid (affine/header from ref)
    base = strip_ext(img_p.name).replace("_T1w_MNI_norm","").replace("_T1w","")
    out_img = OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz"
    out_msk = OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz"

    if not OVERWRITE and out_img.exists() and out_msk.exists():
        if i % 25 == 0 or i == len(pairs):
            print(f"[{i}/{len(pairs)}] (skip, exists) {out_img.name}")
        continue

    save_like(img_ref, xu.astype(np.float32), out_img, dtype=np.float32)
    save_like(img_ref, ensure_uint8_mask(yu), out_msk, dtype=np.uint8)
    wrote += 1

    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] wrote {out_img.name} & {out_msk.name}")

print(f"Done → {OUT_DIR} | wrote {wrote} case(s)")


Discovered unique pairs: 138 (deduped across flat + subfolders)
[10/138] wrote sub-M2086_ses-3021_T1w_MNI_norm.nii.gz & sub-M2086_ses-3021_lesion_mask_MNI_clean.nii.gz
[10/138] wrote sub-M2086_ses-3021_T1w_MNI_norm.nii.gz & sub-M2086_ses-3021_lesion_mask_MNI_clean.nii.gz
[20/138] wrote sub-M2131_ses-472_T1w_MNI_norm.nii.gz & sub-M2131_ses-472_lesion_mask_MNI_clean.nii.gz
[20/138] wrote sub-M2131_ses-472_T1w_MNI_norm.nii.gz & sub-M2131_ses-472_lesion_mask_MNI_clean.nii.gz
[30/138] wrote sub-M2198_ses-1073_T1w_MNI_norm.nii.gz & sub-M2198_ses-1073_lesion_mask_MNI_clean.nii.gz
[30/138] wrote sub-M2198_ses-1073_T1w_MNI_norm.nii.gz & sub-M2198_ses-1073_lesion_mask_MNI_clean.nii.gz
[40/138] wrote sub-M2238_ses-1374_T1w_MNI_norm.nii.gz & sub-M2238_ses-1374_lesion_mask_MNI_clean.nii.gz
[40/138] wrote sub-M2238_ses-1374_T1w_MNI_norm.nii.gz & sub-M2238_ses-1374_lesion_mask_MNI_clean.nii.gz
[50/138] wrote sub-M2306_ses-707_T1w_MNI_norm.nii.gz & sub-M2306_ses-707_lesion_mask_MNI_clean.nii.gz
[50/13

# 4) Reduced SNR (Rician noise)

Why / real-world: Fewer averages / higher acceleration → noisier magnitude images.

What the code does

Normalizes the image to ~[0,1] via robust percentiles.

Estimates foreground mean μ (proxy for “signal”).

Applies signal attenuation α·x and adds Gaussian noise N(0,σ) with
σ ≈ (α·μ)/TARGET_SNR to hit the requested SNR.

Clips to [0,1] and writes with original geometry (masks copied unchanged).

What this mimics

A stable, conservative SNR drop while preserving structure and intensity ordering.

Useful stand-in when you want predictable behavior across datasets.

Why it’s useful

Very robust (hard to break models), no magnitude or FFT artifacts, and keeps the intensity range aligned with training distributions.

Caveats

Not physically perfect (true MR magnitude noise is Rician and depends on coil configuration).

Use this when you want reliability over realism.

In [7]:
# === Reduced SNR via k-space complex noise → magnitude (anchored) ===
from pathlib import Path
import numpy as np

OUT_DIR     = OUT_ROOT / "test_hires_snr_kspace_v1"   # new folder
TARGET_SNR  = 30.0   # start conservative; then try 20; only if stable try 15
SEED        = 123
OVERWRITE   = False
OUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)
pairs = discover_pairs(SRC_DIR)
print(f"Discovered unique pairs: {len(pairs)} (deduped)")

def brain_like_mask(x: np.ndarray) -> np.ndarray:
    # foreground from intensities only (no lesion). Simple & robust.
    if (x > 0).sum() > 1000:
        m = x > 0
    else:
        thr = np.percentile(x, 60.0)
        m = x > thr
    # quick closing to fill small holes
    from scipy.ndimage import binary_closing, generate_binary_structure
    return binary_closing(m, structure=generate_binary_structure(3,1), iterations=1)

def pnorm01(x, p_lo=1, p_hi=99, eps=1e-6):
    lo, hi = np.percentile(x, [p_lo, p_hi])
    if hi - lo < eps: hi = lo + eps
    y = np.clip((x - lo) / (hi - lo), 0, 1)
    return y.astype(np.float32), float(lo), float(hi)

def affine_match(src, ref, mask):
    """Fit y ≈ a*src + b to ref on mask (LS), return a,b."""
    s = src[mask].ravel().astype(np.float64)
    r = ref[mask].ravel().astype(np.float64)
    if s.size < 50:
        return 1.0, 0.0
    A = np.vstack([s, np.ones_like(s)]).T
    a, b = np.linalg.lstsq(A, r, rcond=None)[0]
    if not np.isfinite(a): a = 1.0
    if not np.isfinite(b): b = 0.0
    return float(a), float(b)

def add_kspace_noise_and_magnitude(x_norm, sigma_k):
    """
    1) FFT to k-space, 2) add complex Gaussian noise in k-space,
    3) IFFT back (complex image), 4) take magnitude.
    """
    X = np.fft.fftn(x_norm.astype(np.complex64), norm=None)
    nr = rng.normal(0.0, sigma_k, size=X.shape).astype(np.float32)
    ni = rng.normal(0.0, sigma_k, size=X.shape).astype(np.float32)
    X_noisy = X + (nr + 1j*ni).astype(np.complex64)
    img_complex = np.fft.ifftn(X_noisy, norm=None)
    mag = np.abs(img_complex).astype(np.float32)
    # re-scale mag to ~[0,1] envelope to avoid runaway values from ifft norms
    mag = mag / (np.percentile(mag, 99.5) + 1e-6)
    mag = np.clip(mag, 0, 1)
    return mag

wrote = 0
for i, (img_p, msk_p) in enumerate(pairs, 1):
    img_ref = load_nii(img_p)
    msk_ref = load_nii(msk_p)
    x = data_f32(img_ref); y = data_f32(msk_ref)

    bl = brain_like_mask(x)
    x_norm, x_lo, x_hi = pnorm01(x, 1, 99)

    # choose a k-space noise std that roughly yields desired SNR
    # use foreground mean as "signal" proxy in normalized domain
    mu = float(np.mean(x_norm[bl])) if bl.any() else float(np.mean(x_norm))
    mu = max(mu, 1e-3)
    # empirical mapping: k-space sigma_k produces ~similar image-domain sigma; keep conservative
    sigma_k = np.clip(mu / TARGET_SNR, 1e-4, 0.05)

    x_mag = add_kspace_noise_and_magnitude(x_norm, sigma_k)

    # anchor back to original normalized brain via affine match
    a, b = affine_match(x_mag, x_norm, bl)
    x_adj = np.clip(a * x_mag + b, 0, 1).astype(np.float32)

    # final light clamp (safety) and write
    base = strip_ext(img_p.name).replace("_T1w_MNI_norm","").replace("_T1w","")
    out_img = OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz"
    out_msk = OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz"

    if not OVERWRITE and out_img.exists() and out_msk.exists():
        if i % 25 == 0 or i == len(pairs): print(f"[{i}/{len(pairs)}] (skip) {out_img.name}")
        continue

    save_like(img_ref, x_adj, out_img, dtype=np.float32)
    save_like(msk_ref, (y > 0).astype(np.uint8), out_msk, dtype=np.uint8)
    wrote += 1

    if i % 10 == 0 or i == len(pairs):
        # quick sanity
        corr = np.corrcoef(x_norm[bl].ravel(), x_adj[bl].ravel())[0,1] if bl.any() else np.nan
        print(f"[{i}/{len(pairs)}] wrote {out_img.name} | mu={mu:.3f} sigma_k={sigma_k:.4f} "
              f"| corr(brain)={corr:.3f} | x_adj[min={x_adj.min():.3f}, max={x_adj.max():.3f}, mean={x_adj.mean():.3f}]")

print(f"Done → {OUT_DIR} | wrote {wrote} case(s)")


Discovered unique pairs: 138 (deduped)


[10/138] wrote sub-M2086_ses-3021_T1w_MNI_norm.nii.gz | mu=0.557 sigma_k=0.0186 | corr(brain)=1.000 | x_adj[min=0.000, max=1.000, mean=0.122]
[20/138] wrote sub-M2131_ses-472_T1w_MNI_norm.nii.gz | mu=0.516 sigma_k=0.0172 | corr(brain)=1.000 | x_adj[min=0.000, max=1.000, mean=0.113]
[20/138] wrote sub-M2131_ses-472_T1w_MNI_norm.nii.gz | mu=0.516 sigma_k=0.0172 | corr(brain)=1.000 | x_adj[min=0.000, max=1.000, mean=0.113]
[30/138] wrote sub-M2198_ses-1073_T1w_MNI_norm.nii.gz | mu=0.538 sigma_k=0.0179 | corr(brain)=1.000 | x_adj[min=0.000, max=1.000, mean=0.118]
[30/138] wrote sub-M2198_ses-1073_T1w_MNI_norm.nii.gz | mu=0.538 sigma_k=0.0179 | corr(brain)=1.000 | x_adj[min=0.000, max=1.000, mean=0.118]
[40/138] wrote sub-M2238_ses-1374_T1w_MNI_norm.nii.gz | mu=0.541 sigma_k=0.0180 | corr(brain)=1.000 | x_adj[min=0.000, max=1.000, mean=0.119]
[40/138] wrote sub-M2238_ses-1374_T1w_MNI_norm.nii.gz | mu=0.541 sigma_k=0.0180 | corr(brain)=1.000 | x_adj[min=0.000, max=1.000, mean=0.119]
[50/138]

# 5) Slice-wise motion (rigid jitter)

Why / real-world: Patient motion during 2D acquisitions → slice-to-slice misalignment/blur.

What the code does

For each slice, applies a small random rotation (±a few degrees) and pixel shift (±a few px):
rotate(..., order=1) for image, order=0 for mask; then shift(...) similarly.

This produces slice-to-slice misalignments and slight blurring/ghosting from interpolation.

Voxel spacing unchanged; just geometry perturbations.

What this mimics

2D multi-slice acquisitions where the patient moves between slice excitations → slice stack doesn’t line up perfectly.

Very common in restless patients, pediatrics, or longer scans.

Why it’s useful

Motion is one of the biggest real-world degraders. Even tiny rotations/shift destroy fine boundaries and create zebra-like slice seams.

Caveats

Real motion can be continuous and within-TR; this is a discrete per-slice model (captures the dominant visual effect).

In [3]:
# === 5) SLICE-WISE MOTION JITTER: small per-slice rotations/shifts (XY plane), Z intact ===
from pathlib import Path
import numpy as np
import nibabel as nib
from scipy.ndimage import rotate, shift

OUT_DIR    = OUT_ROOT / "test_hires_motion_slicejitter"
deg_range  = 2.0   # ± degrees of in-plane rotation per slice
px_range   = 2.0   # ± pixels of shift per slice (dy, dx)
SEED       = 7
OVERWRITE  = False

OUT_DIR.mkdir(parents=True, exist_ok=True)

pairs = discover_pairs(SRC_DIR)
print(f"Discovered unique pairs: {len(pairs)} (deduped across flat + subfolders)")

rng   = np.random.default_rng(SEED)
wrote = 0

for i, (img_p, msk_p) in enumerate(pairs, 1):
    img_ref = load_nii(img_p)
    msk_ref = load_nii(msk_p)

    x = data_f32(img_ref)          # (H, W, Z)
    y = np.asarray(msk_ref.get_fdata(), np.float32)
    H, W, Z = x.shape

    # Pre-allocate outputs (same geometry)
    xm = np.empty_like(x, dtype=np.float32)
    ym = np.empty_like(y, dtype=np.float32)

    # Pre-draw all random params for reproducibility across re-runs
    angs = rng.uniform(-deg_range, deg_range, size=Z)
    dxs  = rng.uniform(-px_range,  px_range,  size=Z)
    dys  = rng.uniform(-px_range,  px_range,  size=Z)

    # Per-slice jitter (XY plane), keeping shape via reshape=False
    for k in range(Z):
        ang, dx, dy = float(angs[k]), float(dxs[k]), float(dys[k])

        # Image: rotate (bilinear, order=1), then shift (order=1)
        sl = rotate(x[:, :, k], angle=ang, reshape=False, order=1, mode="nearest")
        sl = shift (sl,        shift=(dy, dx),      order=1, mode="nearest")
        xm[:, :, k] = sl

        # Mask: nearest-neighbor throughout to preserve labels
        slm = rotate(y[:, :, k], angle=ang, reshape=False, order=0, mode="nearest")
        slm = shift (slm,        shift=(dy, dx),      order=0, mode="nearest")
        ym[:, :, k] = slm

    # Optional: clip images back to [0,1] if your pipeline expects that
    # xm = np.clip(xm, 0.0, 1.0, out=xm)

    # Binarize masks after transforms
    ym = (ym > 0.5).astype(np.uint8)

    base = strip_ext(img_p.name).replace("_T1w_MNI_norm","").replace("_T1w","")
    out_img = OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz"
    out_msk = OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz"

    if not OVERWRITE and out_img.exists() and out_msk.exists():
        if i % 25 == 0 or i == len(pairs):
            print(f"[{i}/{len(pairs)}] (skip, exists) {out_img.name}")
        continue

    # Save with original affine/header (same 1mm grid)
    save_like(img_ref, xm, out_img, dtype=np.float32)
    save_like(msk_ref, ym, out_msk, dtype=np.uint8)
    wrote += 1

    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] wrote {out_img.name} & {out_msk.name}")

print(f"Done → {OUT_DIR} | wrote {wrote} case(s)")


Discovered unique pairs: 138 (deduped across flat + subfolders)
[10/138] wrote sub-M2086_ses-3021_T1w_MNI_norm.nii.gz & sub-M2086_ses-3021_lesion_mask_MNI_clean.nii.gz
[10/138] wrote sub-M2086_ses-3021_T1w_MNI_norm.nii.gz & sub-M2086_ses-3021_lesion_mask_MNI_clean.nii.gz
[20/138] wrote sub-M2131_ses-472_T1w_MNI_norm.nii.gz & sub-M2131_ses-472_lesion_mask_MNI_clean.nii.gz
[20/138] wrote sub-M2131_ses-472_T1w_MNI_norm.nii.gz & sub-M2131_ses-472_lesion_mask_MNI_clean.nii.gz
[30/138] wrote sub-M2198_ses-1073_T1w_MNI_norm.nii.gz & sub-M2198_ses-1073_lesion_mask_MNI_clean.nii.gz
[30/138] wrote sub-M2198_ses-1073_T1w_MNI_norm.nii.gz & sub-M2198_ses-1073_lesion_mask_MNI_clean.nii.gz
[40/138] wrote sub-M2238_ses-1374_T1w_MNI_norm.nii.gz & sub-M2238_ses-1374_lesion_mask_MNI_clean.nii.gz
[40/138] wrote sub-M2238_ses-1374_T1w_MNI_norm.nii.gz & sub-M2238_ses-1374_lesion_mask_MNI_clean.nii.gz
[50/138] wrote sub-M2306_ses-707_T1w_MNI_norm.nii.gz & sub-M2306_ses-707_lesion_mask_MNI_clean.nii.gz
[50/13

# 6) k-space undersampling aliasing (zero-filled recon)

Why / real-world: Aggressive parallel imaging / partial-Fourier → residual alias/ghost when reconstruction is imperfect.

What the code does

For each slice (2D FFT), it builds a 1D sampling mask along phase-encode (Y) with:

a fully sampled DC/center band (low frequencies),

and randomly selected outer lines to hit a target acceleration (e.g., 2×).

Sets unacquired lines to zero (no fancy reconstruction), then inverse FFT back.

Geometry unchanged; masks are copied.

What this mimics

Parallel imaging / compressed sensing when acceleration is aggressive or reconstruction fails → residual aliasing/ghosts.

The center band models vendor practice of keeping low-freq fully sampled to preserve contrast.

Why it’s useful

Tests robustness to coherent artifacts (ghosting/alias) that look nothing like Gaussian noise but happen in fast protocols.

Caveats

Real recon uses sensitivity maps and iterative solvers; this is the pessimistic zero-filled limit (harder than reality).

In [7]:
# --- 6) K-SPACE UNDER-SAMPLING: 2x along phase-encode (Y), VD with center fully sampled ---
OUT_DIR = OUT_ROOT / "test_hires_kspace_alias_x2"
rng = np.random.default_rng(42)
accel = 2.0
center_frac = 0.12  # fully-sampled DC region

def undersample_mask(ny, accel, center_frac=0.12):
    m = np.zeros(ny, bool)
    c = int(ny*center_frac/2)
    mid = ny//2
    m[mid-c:mid+c+1] = True
    # pick the rest uniformly at rate 1/accel to reach target
    want = int(round(ny/accel)) - m.sum()
    idxs = [i for i in range(ny) if not m[i]]
    rng.shuffle(idxs)
    m[idxs[:max(want,0)]] = True
    return m

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img)
    H, W, Z = x.shape

    # FFT per slice in XY
    xm = np.empty_like(x)
    for k in range(Z):
        sl = x[:,:,k]
        ksl = np.fft.fftshift(np.fft.fft2(sl))
        mask = undersample_mask(W, accel, center_frac)
        ksl[:, ~mask] = 0
        slr = np.real(np.fft.ifft2(np.fft.ifftshift(ksl)))
        xm[:,:,k] = slr.astype(np.float32)

    base = img_p.stem.replace("_T1w","")
    save_like(img, xm, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz")
    # geometry unchanged → copy mask
    save_like(msk, data_f32(msk), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", dtype=np.uint8)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_kspace_alias_x2


# 7) Gibbs ringing (k-space truncation)

Why / real-world: Limited high-frequency sampling yields ringing near edges (very common on sharp T1).

What the code does

For each slice, FFT to k-space, keep only the central radius (e.g., 75% of k-space energy) and zero the rest.

Inverse FFT back → sharp transitions overshoot/undershoot (ringing near edges).

Geometry unchanged; masks are copied.

What this mimics

Limited high-frequency sampling or apodization issues → Gibbs ringing (very visible on T1 edges).

Also a stand-in for too-aggressive smoothing at recon time.

Why it’s useful

Ringing creates false positives along tissue boundaries and confuses edge-based features learned by CNNs.

Caveats

Real systems use windowing; we emulate a clean radial crop to make the effect consistent and tunable

In [8]:
# --- 7) GIBBS RINGING: crop high-frequency k-space radius per slice ---
OUT_DIR = OUT_ROOT / "test_hires_gibbs"
keep_radius = 0.75  # keep 75% of k-space radius

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img)
    H, W, Z = x.shape
    xm = np.empty_like(x)

    cy, cx = H//2, W//2
    R2 = (min(H, W) * keep_radius / 2.0)**2

    for k in range(Z):
        sl = x[:,:,k]
        ksl = np.fft.fftshift(np.fft.fft2(sl))
        yy, xx = np.ogrid[:H, :W]
        mask = (yy-cy)**2 + (xx-cx)**2 <= R2
        ksl[~mask] = 0
        xm[:,:,k] = np.real(np.fft.ifft2(np.fft.ifftshift(ksl))).astype(np.float32)

    base = img_p.stem.replace("_T1w","")
    save_like(img, xm, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz")
    save_like(msk, data_f32(msk), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", dtype=np.uint8)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_gibbs


# 8) Bias-field inhomogeneity (smooth multiplicative field)

Why / real-world: Receive/B1 inhomogeneity → slow intensity roll-off; can fool intensity normalization.

What the code does

Generates a smooth 3D low-frequency field f(x,y,z) (Gaussian-filtered noise), scaled to be around 1±amp (e.g., ±30%).

Multiplies the image: xb = x * f.

Geometry unchanged; masks are copied.

What this mimics

Receive coil and B1 inhomogeneity → slow intensity roll-off across the head. Stronger at 3T+, surface coils, or large FOVs.

Can break naive intensity normalization and thresholding.

Why it’s useful

Checks whether your preprocessing (e.g., z-score/percentile norm) and the model are tolerant to broad intensity drifts.

Caveats

True bias fields can correlate with anatomy and coil layout; we simulate a generic smooth pattern.

In [9]:
# --- 8) BIAS-FIELD: multiply by smooth low-frequency 3D field ---
OUT_DIR = OUT_ROOT / "test_hires_biasfield"
rng = np.random.default_rng(99)
amp = 0.3  # ±30%
sigma = (40, 40, 20)  # smoothness (vox)

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img)
    # build smooth field ~1±amp
    f = gaussian_filter(rng.normal(0,1,size=x.shape).astype(np.float32), sigma=sigma)
    f = (f - f.min()) / (f.max() - f.min() + 1e-6)
    f = 1.0 + amp*(2*f - 1.0)
    xb = (x * f).astype(np.float32)

    base = img_p.stem.replace("_T1w","")
    save_like(img, xb, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz")
    save_like(msk, data_f32(msk), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", dtype=np.uint8)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_biasfield


# 9) Bit-depth / quantization (8-bit)

Why / real-world: Legacy exports or lossy preprocessing → banding and reduced dynamic range.

What the code does

Scales intensities robustly into [0,1] using image percentiles (e.g., p1–p99).

Quantizes to 256 levels (uint8) and writes that dtype into the NIfTI header.

Geometry unchanged; masks are copied.

What this mimics

Legacy PACS exports, lossy conversion, or “normalized” research datasets that were clamped/quantized.

Banding and loss of subtle gray/white contrast.

Why it’s useful

Tests whether your model truly needs subtle intensity resolution, or if it’s robust to coarse discretization.

Caveats

Real pipelines may apply window/level before quantization; we use percentile scaling to remain dataset-agnostic.

In [2]:
# --- 9) QUANTIZATION to 8-bit using robust scaling (p1–p99) ---
from pathlib import Path
import numpy as np
import nibabel as nib

# --- paths ---
HIRES_DIR = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires")
OUT_ROOT  = HIRES_DIR.parent
OUT_DIR   = OUT_ROOT / "test_hires_quant8"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def niigz_stem(p: Path) -> str:
    """Return filename without the .nii.gz (or last) extension."""
    name = p.name
    if name.endswith(".nii.gz"):
        return name[:-7]
    return p.stem  # fallback

def save_like(ref_path: Path, array: np.ndarray, out_path: Path, dtype=None):
    ref = nib.load(str(ref_path))
    data = np.ascontiguousarray(array.astype(dtype if dtype is not None else np.float32))
    hdr = ref.header.copy()
    nib.save(nib.Nifti1Image(data, ref.affine, hdr), str(out_path))

def robust_uint8_quant(x, p_lo=1, p_hi=99):
    x = np.asarray(x, np.float32)
    lo, hi = np.percentile(x, [p_lo, p_hi])
    if hi <= lo:  # degenerate fallback
        lo, hi = x.min(), x.max() if x.max() > x.min() else (0.0, 1.0)
    y = np.clip((x - lo) / (hi - lo), 0, 1)
    return (y * 255.0 + 0.5).astype(np.uint8)

# find pairs in single-folder layout
imgs = sorted(HIRES_DIR.glob("*_T1w_MNI_norm.nii.gz"))
masks = {niigz_stem(p).replace("_lesion_mask_MNI_clean",""): p
         for p in HIRES_DIR.glob("*_lesion_mask_MNI_clean.nii.gz")}

pairs = []
for img in imgs:
    stem = niigz_stem(img).replace("_T1w_MNI_norm", "")
    msk = masks.get(stem)
    if msk is not None:
        pairs.append((img, msk))

print(f"Found pairs: {len(pairs)}")

for img, msk in pairs:
    # load, quantize to uint8
    ni = nib.load(str(img)); x = ni.get_fdata(dtype=np.float32)
    x8 = robust_uint8_quant(x, p_lo=1, p_hi=99)

    # build sane basenames
    base = niigz_stem(img).replace("_T1w_MNI_norm", "")  # e.g., 'sub-XXX_ses-YYY'
    img_out = OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz"
    msk_out = OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz"

    # write image as uint8 (preserve geometry)
    hdr_img = ni.header.copy(); hdr_img.set_data_dtype(np.uint8)
    nib.save(nib.Nifti1Image(x8, ni.affine, hdr_img), str(img_out))

    # write mask (copy as-is; ensure uint8)
    nm = nib.load(str(msk)); y = nm.get_fdata(dtype=np.float32)
    save_like(msk, (y > 0).astype(np.uint8), msk_out, dtype=np.uint8)

print("Done:", OUT_DIR)


Found pairs: 138
Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_quant8
Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_quant8


# 10) Slice gaps (simulate interleaved acquisition)

Why / real-world: Older 2D protocols with gaps reduce cross-slice context without changing voxel size metadata.

What the code does

Every Nth slice (e.g., 1 in 5), it replaces that slice with a neighbor (keeps array size the same).

This introduces repeated slices and missing anatomical information, mimicking a “gap” without changing pixdim.

Mask slices are duplicated at the same indices.

What this mimics

Older or fast 2D protocols with slice gaps (e.g., 4 mm slice with 1 mm gap). You don’t see anatomy for the skipped positions.

Downstream resampling to isotropic can smear/duplicate planes.

Why it’s useful

A nasty corner case for 3D models: they expect consistent context along Z; gaps violate that assumption.

Caveats

True gaps change physical spacing; here we simulate the visual effect while keeping array dimensions constant for easy downstream processing.

In [5]:
# --- 10) SLICE GAPS: remove every Nth slice and duplicate neighbor to keep size ---
OUT_DIR = OUT_ROOT / "test_hires_slicegap_20pct"
gap_every = 5  # drop 1 in 5 (~20%)

for img_p, msk_p in pairs:
    img = load_nii(img_p); msk = load_nii(msk_p)
    x = data_f32(img); y = data_f32(msk)
    H, W, Z = x.shape
    xm = x.copy(); ym = y.copy()

    for k in range(gap_every-1, Z, gap_every):
        # replace slice k with neighbor (simulate a visual gap)
        src = max(0, k-1)
        xm[:,:,k] = xm[:,:,src]
        ym[:,:,k] = ym[:,:,src]

    base = img_p.stem.replace("_T1w","")
    save_like(img, xm, OUT_DIR / f"{base}_T1w_MNI_norm.nii.gz")
    save_like(msk, ensure_uint8_mask(ym), OUT_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz", dtype=np.uint8)

print("Done:", OUT_DIR)


Done: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires_slicegap_20pct
